# Parda — Phase 4: visual PII detector (YOLO: faces, signatures, QR codes, stamps)

**Settings:** Accelerator **GPU T4** (x1 or x2 — the code pins itself to one GPU) · Internet **On**
**Inputs (Add Input):**
1. **Your Work →** your **Parda** (Phase 1) notebook (the 30,000 synthetic documents)
2. **Datasets →** search **`celeba-dataset`** (by *jessicali9530*, ~1.3 GB). Real face photos that replace the Phase 1
   cartoon avatars, so the detector learns *real* faces. Any big folder of face photos works (e.g. LFW).

**Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`

Run cells 1–7 by hand (~20 min, includes a ~3-minute smoke test). If the smoke test looks right,
**Save Version → Save & Run All** for the real training (hard cap 3 h of training).


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. Fonts + packages (the tests generate a few Hindi/English pages) + ultralytics
!bash scripts/setup_kaggle.sh
!pip install -q ultralytics
!python -c "import ultralytics, torch, cv2; print('ultralytics', ultralytics.__version__, '| opencv', cv2.__version__, '| GPUs:', torch.cuda.device_count(), torch.cuda.get_device_name(0))"


In [ ]:
# 3. Find the synthetic dataset (30000 images) and a folder of real face photos
from parda.ocr.run_ocr import find_data, _n_images
from parda.vision.faces import find_faces_dir
DATA = find_data()
FACES = find_faces_dir("/kaggle/input", exclude=[DATA])
print("DATA  =", DATA, "| images:", _n_images(DATA))
print("FACES =", FACES, "| photos:", len(os.listdir(FACES)) if FACES else 0)
ALLOW_AVATARS = False   # True = train on the Phase 1 cartoon faces (the face detector will NOT work on real photos)
assert FACES or ALLOW_AVATARS, "No face photos found: Add Input -> Datasets -> 'celeba-dataset' (jessicali9530)"
W = "/kaggle/working"


In [ ]:
# 4. Tests (~1 min): every line ok
!python tests/test_vision.py


In [ ]:
# 5. Build the YOLO dataset (~3-5 min): 12000 train pages (all QR pages first), 600 val, the 300 fixed benchmark
#    pages as a held-out test split. Real faces are pasted over the avatars (held-out face photos for val/bench).
faces_arg = f"--faces_dir {FACES}" if FACES else ""
!python -m parda.vision.yolo_data --data {DATA} --out {W}/yolo_data --n_train 12000 --n_val 600 {faces_arg} --workers 4 --preview 4
import glob
from IPython.display import Image as IPImage, Markdown, display
for f in sorted(glob.glob(f"{W}/yolo_data/previews/*.jpg"))[:4]:
    print(os.path.basename(f)); display(IPImage(f, width=420))


In [ ]:
# 6. BEFORE: classic OpenCV (Haar faces + QR detector; nothing for signatures/stamps) on the 300 benchmark pages (~2 min)
!python -m parda.vision.evaluate_vis --yolo_data {W}/yolo_data --systems opencv --out {W}/vis_eval_before > /dev/null
display(Markdown(open(f"{W}/vis_eval_before/report_vis.md").read()))


In [ ]:
# 7. SMOKE TEST (~3 min): 1 epoch on 5% of the training pages. Check: losses are numbers (not nan),
#    "epochs_run": 1, and a best.pt was written. est_full_epoch_min is a rough upper bound per full epoch.
!python -m parda.vision.train_yolo --data_yaml {W}/yolo_data/data.yaml --out /tmp/yolo_smoke --smoke 2>&1 | tail -40


In [ ]:
# 8. REAL TRAINING: up to 40 epochs, hard-capped at 3 h (ultralytics re-plans the epochs to fit the time).
#    Saves /kaggle/working/yolo/best.pt and uploads it to a private HF repo.
from huggingface_hub import HfApi
HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
!python -m parda.vision.train_yolo --data_yaml {W}/yolo_data/data.yaml --out {W}/yolo --epochs 40 --hours 3 --hf_repo {HF_USER}/parda-yolo-v1


In [ ]:
# 9. AFTER: OpenCV baseline vs the trained YOLO on the 300 benchmark pages (~3 min)
!python -m parda.vision.evaluate_vis --yolo_data {W}/yolo_data --systems opencv,yolo --weights {W}/yolo/best.pt --thresholds 0.25,0.5 --out {W}/vis_eval_after > /dev/null
display(Markdown(open(f"{W}/vis_eval_after/report_vis.md").read()))


In [ ]:
# 10. Collect results to commit to GitHub (download the folder from the Output tab)
R = f"{W}/phase4_results"
!mkdir -p {R}
!cp {W}/vis_eval_before/report_vis.md {R}/report_vis_before.md
!cp {W}/vis_eval_after/report_vis.md {W}/vis_eval_after/report_vis.json {W}/yolo/train_summary.json {R}/
!cp {W}/yolo_data/stats.json {R}/data_stats.json
!cd {W}/yolo/train && cp results.csv results.png confusion_matrix_normalized.png val_batch0_pred.jpg {R}/ 2>/dev/null; true
!cp {W}/yolo_data/previews/*.jpg {R}/ 2>/dev/null; true
!ls -la {R}
